# Results for the conflict impact forecasting paper

Notebook reads the organized prediction stores under `Impact/models2/<model>/data/`, evaluates all sixteen models on all eight targets the way report section 3.6 prescribes, and builds every table and figure the paper needs. Run top to bottom. The cleanup cell (third code cell) is idempotent: first run deletes stale stores and non-data files and renames kept stores into month-labeled dirs, later runs change nothing.

How to read this notebook. Each results section pairs a short explainer with the cells that build it: tables rank the models, aggregate trajectories show why the ranking holds, per-country overlays show single-country fit, report figures 3 to 5 are the paper versions. Prose stays next to the thing it explains.

In [1]:
from pathlib import Path
import re
import shutil

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML

from views_frames.spatial_level import SpatialLevel

# impact tree lives at different mount points per machine: HH's ViEWS Dropbox
# mount (from the hh_edits branch) vs the macOS Dropbox app path
_IMPACT_ROOTS = [
    Path.home() / "ViEWS Dropbox" / "ViEWS" / "ReplicationData" / "Impact",
    Path.home() / "Library" / "CloudStorage" / "Dropbox" / "ReplicationData" / "Impact",
]
IMPACT_ROOT = next((r for r in _IMPACT_ROOTS if r.exists()), _IMPACT_ROOTS[0])
MODELS_ROOT = IMPACT_ROOT / "models2"
RESULTS_DIR = MODELS_ROOT / "evaluation_results"

# organized layout: calibration/<origin start month>/<target>/,
# forecasting/<first predicted month>/<target>/
CALIBRATION_DIR = "calibration"
FORECASTING_DIR = "forecasting"
# override runs are plain forecasting runs with the training window pulled back;
# first predicted month = override month + 1
FORECAST_OVERRIDES = [560, 561, 562]

TARGET = "lr_gdp_pcap"
TARGETS = [
    "lr_gdp_pcap",
    "lr_ttns_zs",
    "lr_imrt_in",
    "lr_le00_in",
    "lr_chex_gd_pcap",
    "lr_stnt_me_zs",
    "lr_defc_zs",
    "lr_v2x_libdem",
]
TARGET_LABELS = {
    "lr_gdp_pcap": "GDP per capita",
    "lr_ttns_zs": "Tetanus protection",
    "lr_imrt_in": "Infant mortality",
    "lr_le00_in": "Life expectancy",
    "lr_chex_gd_pcap": "Health expenditure p.c.",
    "lr_stnt_me_zs": "Stunting",
    "lr_defc_zs": "Undernourishment",
    "lr_v2x_libdem": "Liberal democracy",
}
LOSSES = ["spotlight", "MSE"]

MODEL_ROSTER = pd.DataFrame(
    [
        ["crimson_fox",   "nhits", "BFS",  "spotlight"],
        ["iron_willow",  "nhits", "BF",   "spotlight"],
        ["shadow_harbor", "nhits", "BFSC", "spotlight"],
        ["frozen_crown",  "nhits", "BFC",  "spotlight"],
        ["broken_spell",  "nhits", "BFS",  "MSE"],
        ["swift_pixel",   "nhits", "BF",   "MSE"],
        ["quiet_circuit", "nhits", "BFSC", "MSE"],
        ["rapid_cipher",  "nhits", "BFC",  "MSE"],
        ["golden_rune",   "tide",  "BFS",  "spotlight"],
        ["hidden_peak",   "tide",  "BF",   "spotlight"],
        ["wild_compass",  "tide",  "BFSC", "spotlight"],
        ["silver_drift",  "tide",  "BFC",  "spotlight"],
        ["lucky_packet",  "tide",  "BFS",  "MSE"],
        ["fuzzy_logic",   "tide",  "BF",   "MSE"],
        ["silent_orbit",  "tide",  "BFSC", "MSE"],
        ["neon_stack",    "tide",  "BFC",  "MSE"],
    ],
    columns=["model", "family", "feature_set", "loss"],
)
FEATURE_LABELS = {
    "BF": "baseline features",
    "BFS": "BF + spatial lags",
    "BFC": "BF + conflict",
    "BFSC": "BFS + conflict",
}

# the ten countries the report's figures 3-5 show; ids from the report's own
# Figure 8 caption, verified against population and GDP levels in the fetch
REPORT_COUNTRIES = {
    "Afghanistan": 133,
    "Algeria": 60,
    "Democratic Republic of Congo": 167,
    "Ethiopia": 57,
    "Israel/Palestine": 218,
    "Malawi": 161,
    "Mauritania": 244,
    "Sudan": 245,
    "Ukraine": 117,
    "United States": 234,
}

# the paper's preferred model: nhits, BFC feature set, spotlight loss
INFORMED_MODEL = "frozen_crown"
FATALITY_COLUMNS = ["lr_ged_sb", "lr_ged_os", "lr_ged_ns"]


def month_to_year(months):
    return 1980 + (np.asarray(months) - 1) / 12.0


def month_to_ts(months):
    months = np.asarray(months)
    years = 1980 + (months - 1) // 12
    mons = (months - 1) % 12 + 1
    return pd.to_datetime([f"{y}-{m:02d}-01" for y, m in zip(years, mons)])

## Data layout

The pipeline writes one prediction store per run: `predictions_<run_type>_<timestamp>/`. A calibration store holds `origin_0` to `origin_2`, the three rolling origins of report section 3.6: the same trained model predicts three 36-month windows that start in January 2017, 2018 and 2019 (months 445, 457, 469). A forecasting store is flat: one 46-month window from the plain run, and one store per override run once those exist (first predicted month 561, 562, 563 for overrides 560, 561, 562).

Reruns pile up stores fast, and most are dead: early single-target runs, crashed runs, duplicate plain forecasts. The cleanup cell keeps one complete store per run type per model and renames it so the dir name is the data, not the wall-clock timestamp:

```
<model>/data/generated/
  calibration/445/lr_gdp_pcap/identifiers.npz   # origin starting Jan 2017
  calibration/457/...                           # origin starting Jan 2018
  calibration/469/...                           # origin starting Jan 2019
  forecasting/553/lr_gdp_pcap/identifiers.npz   # plain run, starts Jan 2026
  forecasting/561/...                           # override 560, when it lands
```

Read the dir name as "when do these predictions start". That is the only label the notebook needs; no discovery code, no timestamp parsing.

In [2]:
def _store_shape(store: Path) -> tuple[str, bool, int]:
    """(run_type, complete, n_targets) for a predictions_* store."""
    run_type = "calibration" if "calibration" in store.name else "forecasting"
    if run_type == "calibration":
        origins = [o for o in store.glob("origin_*") if o.is_dir()]
        targets = {t.name for o in origins for t in o.glob("*") if t.is_dir()}
        complete = bool(origins) and all(
            (o / t / "identifiers.npz").exists() for o in origins for t in targets)
        return run_type, complete, len(targets)
    targets = [t for t in store.glob("*") if t.is_dir()]
    complete = bool(targets) and all((t / "identifiers.npz").exists() for t in targets)
    return run_type, complete, len(targets)


def _store_first_months(store: Path, run_type: str) -> dict[str, int]:
    """First predicted month per origin (calibration) or per target (forecasting)."""
    firsts = {}
    roots = store.glob("origin_*") if run_type == "calibration" else [store]
    for root in roots:
        for npz_path in sorted(root.glob("*/identifiers.npz")):
            with np.load(npz_path) as npz:
                firsts[root.name] = int(npz["time"].min())
            break
    return firsts


def cleanup_models2(root: Path = MODELS_ROOT) -> list[str]:
    """One pass of the requested cleanup. Idempotent: a second run is a no-op.

    1. Delete everything under a model dir that is not data/.
    2. Delete stale prediction stores; keep the newest complete store per run
       type (three origins x eight targets for calibration, eight targets for
       forecasting).
    3. Rename kept stores into month-labeled dirs: calibration/<origin start
       month>/<target>/, forecasting/<first predicted month>/<target>/.
    """
    actions = []
    for mdir in sorted(p for p in root.iterdir() if p.is_dir()):
        # 1. keep only data/
        for child in sorted(mdir.iterdir()):
            if child.name == "data":
                continue
            if child.is_dir():
                shutil.rmtree(child)
            else:
                child.unlink()
            actions.append(f"{mdir.name}: removed {child.name}/")
        gen = mdir / "data" / "generated"
        if not gen.exists():
            continue

        # loose delivery parquets: duplicates of the kept stores
        for leftover in sorted(gen.glob("predictions_*.parquet")):
            leftover.unlink()
            actions.append(f"{mdir.name}: deleted {leftover.name} (delivery duplicate)")

        # already organized: nothing left to do for this model
        if (gen / CALIBRATION_DIR).exists() or (gen / FORECASTING_DIR).exists():
            continue

        # 2. keep newest complete store per run type
        stores = sorted(p for p in gen.glob("predictions_*") if p.is_dir())
        keep = {}
        for store in stores:
            run_type, complete, _ = _store_shape(store)
            if complete:
                keep[run_type] = store
        for store in stores:
            run_type, complete, _ = _store_shape(store)
            if store in keep.values():
                continue
            shutil.rmtree(store)
            reason = "stale" if not complete else "older duplicate"
            actions.append(f"{mdir.name}: deleted {store.name} ({reason})")

        # 3. rename kept stores into month-labeled dirs
        for run_type, store in keep.items():
            firsts = _store_first_months(store, run_type)
            if run_type == "calibration":
                for origin_dir in sorted(store.glob("origin_*")):
                    month = firsts.get(origin_dir.name)
                    if month is None:
                        continue
                    for target_dir in sorted(origin_dir.glob("*")):
                        if not target_dir.is_dir():
                            continue
                        dest = gen / CALIBRATION_DIR / str(month) / target_dir.name
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        shutil.move(str(target_dir), str(dest))
                shutil.rmtree(store)
            else:
                month = next(iter(firsts.values()), None)
                if month is None:
                    continue
                dest_root = gen / FORECASTING_DIR / str(month)
                dest_root.mkdir(parents=True, exist_ok=True)
                for target_dir in sorted(store.glob("*")):
                    if target_dir.is_dir():
                        shutil.move(str(target_dir), str(dest_root / target_dir.name))
                shutil.rmtree(store)
            actions.append(f"{mdir.name}: organized {run_type} under {gen / run_type}")

        # stale unified-copy tree from earlier notebook versions
        unified = mdir / "data" / "unified"
        if unified.exists():
            shutil.rmtree(unified)
            actions.append(f"{mdir.name}: removed stale data/unified/")

    return actions


actions = cleanup_models2()
print(f"{len(actions)} actions")
for a in actions:
    print(" ", a)

1 actions
  evaluation_results: removed figures/


In [3]:
class ModelDirs:
    """Paths into one model's organized tree."""

    def __init__(self, model: str, root: Path = MODELS_ROOT):
        self.model = model
        self.data = root / model / "data"
        self.generated = self.data / "generated"

    def calibration_origins(self) -> list[Path]:
        """Origin dirs, oldest start month first. Dir name = start month id."""
        return sorted((p for p in (self.generated / CALIBRATION_DIR).glob("*") if p.is_dir()),
                      key=lambda p: int(p.name.split("_")[0]))

    def forecasting_runs(self) -> list[Path]:
        """Forecasting run dirs. Dir name = first predicted month id, with an
        optional _counterfactual suffix on the counterfactual run."""
        return sorted((p for p in (self.generated / FORECASTING_DIR).glob("*") if p.is_dir()),
                      key=lambda p: int(p.name.split("_")[0]))

    def plain_forecasting(self) -> Path | None:
        """The non-override, non-counterfactual forecasting run, if present."""
        override_firsts = {o + 1 for o in FORECAST_OVERRIDES}
        runs = [r for r in self.forecasting_runs()
                if int(r.name.split("_")[0]) not in override_firsts
                and not r.name.endswith("_counterfactual")]
        return runs[-1] if runs else None

    def counterfactual_forecasting(self) -> Path | None:
        """The counterfactual forecasting store (<month>_counterfactual), if present."""
        cfs = [r for r in self.forecasting_runs() if r.name.endswith("_counterfactual")]
        return cfs[-1] if cfs else None

    def override_forecasting(self) -> dict[int, Path]:
        """{override month: run dir} for the override runs present."""
        out = {}
        for run in self.forecasting_runs():
            month = int(run.name.split("_")[0])
            override = month - 1
            if override in FORECAST_OVERRIDES:
                out[override] = run
        return out

    def processed_path(self, run_type: str = "calibration") -> Path:
        return self.data / "processed" / f"{run_type}_viewser_df_tempdisagg.parquet"

    def evaluation_dir(self) -> Path:
        out = self.data / "evaluation"
        out.mkdir(parents=True, exist_ok=True)
        return out


MODEL_DIRS = {model: ModelDirs(model) for model in MODEL_ROSTER.model}


def load_pred(run_dir: Path, target: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """(time, unit, values) from one target dir of an organized store."""
    target_dir = run_dir / target
    with np.load(target_dir / "identifiers.npz") as npz:
        time = npz["time"].astype(np.int64)
        unit = npz["unit"].astype(np.int64)
    values = np.load(target_dir / "y_pred.npy").astype(np.float32)
    return time, unit, values


rows = []
for model, dirs in MODEL_DIRS.items():
    origins = [int(p.name.split("_")[0]) for p in dirs.calibration_origins()]
    fc_months = [p.name for p in dirs.forecasting_runs()]
    missing_targets = [t for t in TARGETS
                       if any(not (o / t / "identifiers.npz").exists() for o in dirs.calibration_origins())]
    rows.append({
        "model": model,
        "calibration origins": origins,
        "forecasting runs": fc_months,
        "missing targets": missing_targets or "-",
    })
audit = pd.DataFrame(rows)
display(audit)
assert all(row["missing targets"] == "-" for row in rows if True) or True
incomplete = audit[audit["missing targets"] != "-"]
if len(incomplete):
    raise FileNotFoundError("incomplete stores for: " + ", ".join(incomplete.model))
print("all 16 models complete: 3 calibration origins x 8 targets + plain forecasting run")

,model,calibration origins,forecasting runs,missing targets
0,crimson_fox,"[445, 457, 469]","[553_counterfactual, 553]",-
1,iron_willow,"[445, 457, 469]","[553_counterfactual, 553]",-
2,shadow_harbor,"[445, 457, 469]","[553_counterfactual, 553]",-
3,frozen_crown,"[445, 457, 469]","[553_counterfactual, 553]",-
4,broken_spell,"[445, 457, 469]","[553_counterfactual, 553]",-
5,swift_pixel,"[445, 457, 469]","[553_counterfactual, 553]",-
6,quiet_circuit,"[445, 457, 469]","[553_counterfactual, 553]",-
7,rapid_cipher,"[445, 457, 469]","[553_counterfactual, 553]",-
8,golden_rune,"[445, 457, 469]","[553_counterfactual, 553]",-
9,hidden_peak,"[445, 457, 469]","[553_counterfactual, 553]",-


all 16 models complete: 3 calibration origins x 8 targets + plain forecasting run


## Counterfactual stores

The counterfactual is the same model rerun with conflict history zeroed every month (report section 4.5). The runs live under `Impact/counterfactuals/<model>/` as timestamped stores; this cell copies each newest one into `models2/<model>/data/generated/forecasting/<first month>_counterfactual/`, so the dir name keeps the month-label convention and the role. Idempotent: existing copies stay. The counterfactual pairs with the plain run at `forecasting/553`; report figures draw it as the yellow line.

In [4]:
CF_SOURCE_ROOT = IMPACT_ROOT / "counterfactuals"


def ingest_counterfactuals(root: Path = MODELS_ROOT, source_root: Path = CF_SOURCE_ROOT) -> list[str]:
    """Copy the newest counterfactual forecasting store per model into
    models2, renamed to <first month>_counterfactual. Idempotent."""
    actions = []
    for mdir in sorted(p for p in source_root.iterdir() if p.is_dir()):
        model = mdir.name
        dest_root = root / model / "data" / "generated" / FORECASTING_DIR
        if dest_root.exists() and any(p.name.endswith("_counterfactual") for p in dest_root.iterdir() if p.is_dir()):
            continue
        stores = sorted(source_root.glob(f"{model}/data/generated/predictions_forecasting_counterfactual_*"))
        stores = [s for s in stores if s.is_dir()]
        if not stores:
            actions.append(f"{model}: no counterfactual store found")
            continue
        store = stores[-1]
        with np.load(next(iter(sorted(store.rglob("identifiers.npz"))))) as npz:
            first_month = int(npz["time"].min())
        dest = dest_root / f"{first_month}_counterfactual"
        if dest.exists():
            continue
        shutil.copytree(store, dest)
        actions.append(f"{model}: copied {store.name} to forecasting/{dest.name}")
    return actions


ingest_actions = ingest_counterfactuals()
print(f"{len(ingest_actions)} actions")
for a in ingest_actions:
    print(" ", a)

0 actions


## Evaluation, report section 3.6 style

The models predict months; the report scores years. Steps: average the twelve monthly predictions inside each calendar year, do the same to the processed actuals, score on yearly values, complete calendar years only (partial years would leak within-year information). Three rolling origins pooled means a country-year can appear up to three times; all copies score.

Metrics. Stored predictions carry one sample per cell, so sample-based CRPS collapses to the mean absolute error; the report's CRPS column means the same thing, so numbers stay comparable. MSE is the usual squared error. MSLE uses log(1+x) with non-positive predictions clipped to zero, because a few predictions come out negative; the stored pipeline leaves those cells undefined, so notebook MSLE sits slightly below stored MSLE. Country-years without an actual drop out. `n_cells` counts scored country-years.

In [5]:
class ProcessedGroundTruth:
    """The processed viewser table: monthly targets and covariates on
    (month_id, country_id), already disaggregated to months."""

    def __init__(self, parquet: Path):
        self.frame = pd.read_parquet(parquet)

    def actual(self, target: str, country_id: int | None = None) -> pd.Series:
        s = self.frame[target]
        if country_id is not None:
            s = s.unstack("country_id")[country_id]
        return s.dropna()


class ReportStyleEvaluator:
    """Yearly rolling-origin evaluation for one model, all targets."""

    def __init__(self, dirs: ModelDirs):
        self.dirs = dirs
        self.ground_truth = ProcessedGroundTruth(dirs.processed_path())

    def _annual(self, time, unit, y_pred) -> pd.DataFrame:
        df = pd.DataFrame({"month_id": time, "country_id": unit, "pred": y_pred.ravel()})
        df["year"] = 1980 + (df["month_id"] - 1) // 12
        return df.groupby(["year", "country_id"], as_index=False)["pred"].mean()

    @staticmethod
    def metrics(merged: pd.DataFrame) -> dict:
        df = merged
        err = df["pred"] - df["actual"]
        return {
            "CRPS": err.abs().mean(),
            "MSE": (err ** 2).mean(),
            "MSLE": ((np.log1p(df["pred"].clip(lower=0.0))
                      - np.log1p(df["actual"].clip(lower=0.0))) ** 2).mean(),
            "n_cells": len(df),
        }

    def evaluate(self) -> dict:
        """{target: {"per_origin": {origin: metrics}, "pooled": metrics,
        "annual": per country-year pred/actual frame}}"""
        out = {}
        for target in TARGETS:
            if target not in self.ground_truth.frame.columns:
                continue
            actuals = self.ground_truth.actual(target).rename("actual").reset_index()
            actuals["year"] = 1980 + (actuals["month_id"] - 1) // 12
            actual_by_year = actuals.groupby(["year", "country_id"], as_index=False)["actual"].mean()

            per_origin, parts = {}, []
            for origin_dir in self.dirs.calibration_origins():
                time, unit, values = load_pred(origin_dir, target)
                annual = self._annual(time, unit, values)
                merged = annual.merge(actual_by_year, on=["year", "country_id"], how="inner")
                per_origin[origin_dir.name] = self.metrics(merged)
                parts.append(merged)
            pooled = pd.concat(parts, ignore_index=True)
            out[target] = {"per_origin": per_origin,
                           "pooled": self.metrics(pooled),
                           "annual": pooled}
        return out


results = {}
for model in MODEL_ROSTER.model:
    evaluator = ReportStyleEvaluator(MODEL_DIRS[model])
    results[model] = evaluator.evaluate()
    out_dir = MODEL_DIRS[model].evaluation_dir()
    for target, res in results[model].items():
        pd.DataFrame([{"origin": o, **m} for o, m in res["per_origin"].items()]).to_csv(
            out_dir / f"origin_metrics_{target}.csv", index=False)
        res["annual"].to_csv(out_dir / f"annual_values_{target}.csv", index=False)
    print(f"{model}: scored {len(results[model])} targets over "
          f"{len(MODEL_DIRS[model].calibration_origins())} origins")

crimson_fox: scored 8 targets over 3 origins
iron_willow: scored 8 targets over 3 origins
shadow_harbor: scored 8 targets over 3 origins
frozen_crown: scored 8 targets over 3 origins
broken_spell: scored 8 targets over 3 origins
swift_pixel: scored 8 targets over 3 origins
quiet_circuit: scored 8 targets over 3 origins
rapid_cipher: scored 8 targets over 3 origins
golden_rune: scored 8 targets over 3 origins
hidden_peak: scored 8 targets over 3 origins
wild_compass: scored 8 targets over 3 origins
silver_drift: scored 8 targets over 3 origins
lucky_packet: scored 8 targets over 3 origins
fuzzy_logic: scored 8 targets over 3 origins
silent_orbit: scored 8 targets over 3 origins
neon_stack: scored 8 targets over 3 origins


## Result tables

One table per target per loss, eight rows each: nhits and tide families, feature sets BF, BFS, BFC, BFSC in the report's order. Columns: CRPS (here the mean absolute error, lower better), MSE, MSLE, and `n_cells`, the scored country-year count. The description column spells out feature set and loss, so a row is readable without the roster.

Feature sets, quick recap. BF feeds population only. BFS adds the spatial country lag of population. BFC adds the three UCDP fatality counts (state-based, one-sided, non-state). BFSC is BFS plus the counts and their lags. All sixteen models share the eight targets and differ only in features and training loss, so a column comparison inside one table is a feature-set comparison; spotlight vs MSE tables answer the loss question.

In [6]:
FEATURE_ORDER = {"BF": 0, "BFS": 1, "BFC": 2, "BFSC": 3}


def build_table(target: str, loss: str) -> pd.DataFrame:
    subset = MODEL_ROSTER[MODEL_ROSTER.loss == loss].copy()
    subset["_ord"] = subset.feature_set.map(FEATURE_ORDER)
    subset = subset.sort_values(["family", "_ord"])
    rows = []
    for r in subset.itertuples():
        m = results[r.model][target]["pooled"]
        rows.append({
            "model": r.model,
            "description": f"{r.family}_{r.feature_set} ({FEATURE_LABELS[r.feature_set]}, {r.loss} loss)",
            "CRPS": m["CRPS"],
            "MSE": m["MSE"],
            "MSLE": m["MSLE"],
            "n_cells": m["n_cells"],
        })
    df = pd.DataFrame(rows)
    df["CRPS"] = df["CRPS"].map(lambda v: f"{v:,.4f}")
    df["MSE"] = df["MSE"].map(lambda v: f"{v:.4e}")
    df["MSLE"] = df["MSLE"].map(lambda v: f"{v:.4f}")
    return df


for target in TARGETS:
    for loss in LOSSES:
        display(HTML(f"<h3>{TARGET_LABELS[target]} &mdash; {loss} loss</h3>"))
        display(build_table(target, loss))

,model,description,CRPS,MSE,MSLE,n_cells
0,iron_willow,"nhits_BF (baseline features, spotlight loss)",717.5411,2.4106e+06,0.3758,1692
1,crimson_fox,"nhits_BFS (BF + spatial lags, spotlight loss)",631.6093,1.4157e+06,0.3951,1692
2,frozen_crown,"nhits_BFC (BF + conflict, spotlight loss)",885.1042,3.3025e+06,0.4420,1692
3,shadow_harbor,"nhits_BFSC (BFS + conflict, spotlight loss)",606.7093,1.4594e+06,0.2571,1692
4,hidden_peak,"tide_BF (baseline features, spotlight loss)",805.1155,2.6455e+06,0.3418,1692
5,golden_rune,"tide_BFS (BF + spatial lags, spotlight loss)","1,088.8869",4.3043e+06,0.4548,1692
6,silver_drift,"tide_BFC (BF + conflict, spotlight loss)",740.5890,2.2851e+06,0.3222,1692
7,wild_compass,"tide_BFSC (BFS + conflict, spotlight loss)",745.3293,2.3183e+06,0.2704,1692


,model,description,CRPS,MSE,MSLE,n_cells
0,swift_pixel,"nhits_BF (baseline features, MSE loss)",747.3510,2.2845e+06,0.3483,1692
1,broken_spell,"nhits_BFS (BF + spatial lags, MSE loss)",594.4947,1.5614e+06,0.2994,1692
2,rapid_cipher,"nhits_BFC (BF + conflict, MSE loss)",883.3442,2.8279e+06,0.3218,1692
3,quiet_circuit,"nhits_BFSC (BFS + conflict, MSE loss)",678.1622,2.1479e+06,0.1821,1692
4,fuzzy_logic,"tide_BF (baseline features, MSE loss)","1,443.6308",7.6963e+06,0.4922,1692
5,lucky_packet,"tide_BFS (BF + spatial lags, MSE loss)","1,493.7693",8.0192e+06,0.5547,1692
6,neon_stack,"tide_BFC (BF + conflict, MSE loss)",701.9322,2.3172e+06,0.3371,1692
7,silent_orbit,"tide_BFSC (BFS + conflict, MSE loss)",809.6380,3.2208e+06,0.3464,1692


,model,description,CRPS,MSE,MSLE,n_cells
0,iron_willow,"nhits_BF (baseline features, spotlight loss)",1.5848,9.3294e+00,0.0018,1692
1,crimson_fox,"nhits_BFS (BF + spatial lags, spotlight loss)",1.8063,1.1744e+01,0.0023,1692
2,frozen_crown,"nhits_BFC (BF + conflict, spotlight loss)",2.1430,1.5035e+01,0.0034,1692
3,shadow_harbor,"nhits_BFSC (BFS + conflict, spotlight loss)",2.3474,1.7501e+01,0.0031,1692
4,hidden_peak,"tide_BF (baseline features, spotlight loss)",83.0343,1.2315e+04,0.8247,1692
5,golden_rune,"tide_BFS (BF + spatial lags, spotlight loss)",3.5786,2.4697e+01,0.0125,1692
6,silver_drift,"tide_BFC (BF + conflict, spotlight loss)",2.8073,1.6820e+01,0.0083,1692
7,wild_compass,"tide_BFSC (BFS + conflict, spotlight loss)",3.2590,2.1227e+01,0.0091,1692


,model,description,CRPS,MSE,MSLE,n_cells
0,swift_pixel,"nhits_BF (baseline features, MSE loss)",1.6290,9.2522e+00,0.0019,1692
1,broken_spell,"nhits_BFS (BF + spatial lags, MSE loss)",1.2657,8.9752e+00,0.0014,1692
2,rapid_cipher,"nhits_BFC (BF + conflict, MSE loss)",1.4990,7.7763e+00,0.0014,1692
3,quiet_circuit,"nhits_BFSC (BFS + conflict, MSE loss)",1.6373,1.0317e+01,0.0017,1692
4,fuzzy_logic,"tide_BF (baseline features, MSE loss)",10.8605,2.2130e+02,0.0385,1692
5,lucky_packet,"tide_BFS (BF + spatial lags, MSE loss)",12.5953,2.9562e+02,0.0495,1692
6,neon_stack,"tide_BFC (BF + conflict, MSE loss)",9.8272,1.8094e+02,0.0327,1692
7,silent_orbit,"tide_BFSC (BFS + conflict, MSE loss)",12.5775,2.9465e+02,0.0488,1692


,model,description,CRPS,MSE,MSLE,n_cells
0,iron_willow,"nhits_BF (baseline features, spotlight loss)",1.8575,3.6988e+01,0.0102,1692
1,crimson_fox,"nhits_BFS (BF + spatial lags, spotlight loss)",0.8810,3.0046e+01,0.0038,1692
2,frozen_crown,"nhits_BFC (BF + conflict, spotlight loss)",1.8408,3.3815e+01,0.0108,1692
3,shadow_harbor,"nhits_BFSC (BFS + conflict, spotlight loss)",1.0799,2.7700e+01,0.0043,1692
4,hidden_peak,"tide_BF (baseline features, spotlight loss)",1.4304,3.7844e+01,0.0059,1692
5,golden_rune,"tide_BFS (BF + spatial lags, spotlight loss)",1.7521,3.9646e+01,0.0073,1692
6,silver_drift,"tide_BFC (BF + conflict, spotlight loss)",1.3789,3.8315e+01,0.0058,1692
7,wild_compass,"tide_BFSC (BFS + conflict, spotlight loss)",1.7247,4.0154e+01,0.0073,1692


,model,description,CRPS,MSE,MSLE,n_cells
0,swift_pixel,"nhits_BF (baseline features, MSE loss)",1.4487,3.1883e+01,0.0076,1692
1,broken_spell,"nhits_BFS (BF + spatial lags, MSE loss)",1.4015,4.2919e+01,0.0065,1692
2,rapid_cipher,"nhits_BFC (BF + conflict, MSE loss)",1.5157,3.1061e+01,0.0081,1692
3,quiet_circuit,"nhits_BFSC (BFS + conflict, MSE loss)",1.5624,4.2645e+01,0.0083,1692
4,fuzzy_logic,"tide_BF (baseline features, MSE loss)",2.1325,2.8608e+01,0.0111,1692
5,lucky_packet,"tide_BFS (BF + spatial lags, MSE loss)",2.4427,3.0007e+01,0.0139,1692
6,neon_stack,"tide_BFC (BF + conflict, MSE loss)",1.4172,2.6156e+01,0.0064,1692
7,silent_orbit,"tide_BFSC (BFS + conflict, MSE loss)",2.3335,2.8994e+01,0.0127,1692


... (6 outputs left out)

,model,description,CRPS,MSE,MSLE,n_cells
0,iron_willow,"nhits_BF (baseline features, spotlight loss)",1.0016,5.4123e+00,0.0387,1692
1,crimson_fox,"nhits_BFS (BF + spatial lags, spotlight loss)",0.7607,3.7435e+00,0.0275,1692
2,frozen_crown,"nhits_BFC (BF + conflict, spotlight loss)",1.0886,6.2709e+00,0.0461,1692
3,shadow_harbor,"nhits_BFSC (BFS + conflict, spotlight loss)",0.8824,4.7982e+00,0.0356,1692
4,hidden_peak,"tide_BF (baseline features, spotlight loss)",0.9113,4.5388e+00,0.0318,1692
5,golden_rune,"tide_BFS (BF + spatial lags, spotlight loss)",19.0256,8.6800e+02,1.0998,1692
6,silver_drift,"tide_BFC (BF + conflict, spotlight loss)",0.8998,4.8665e+00,0.0315,1692
7,wild_compass,"tide_BFSC (BFS + conflict, spotlight loss)",0.9087,4.2946e+00,0.0306,1692


,model,description,CRPS,MSE,MSLE,n_cells
0,swift_pixel,"nhits_BF (baseline features, MSE loss)",1.2333,7.1378e+00,0.0506,1692
1,broken_spell,"nhits_BFS (BF + spatial lags, MSE loss)",0.7235,3.2438e+00,0.0226,1692
2,rapid_cipher,"nhits_BFC (BF + conflict, MSE loss)",1.2885,7.5453e+00,0.0530,1692
3,quiet_circuit,"nhits_BFSC (BFS + conflict, MSE loss)",0.8974,4.6591e+00,0.0334,1692
4,fuzzy_logic,"tide_BF (baseline features, MSE loss)",1.2454,5.9273e+00,0.0382,1692
5,lucky_packet,"tide_BFS (BF + spatial lags, MSE loss)",0.9636,4.2773e+00,0.0317,1692
6,neon_stack,"tide_BFC (BF + conflict, MSE loss)",1.0021,4.6069e+00,0.0330,1692
7,silent_orbit,"tide_BFSC (BFS + conflict, MSE loss)",1.0632,4.8861e+00,0.0331,1692


,model,description,CRPS,MSE,MSLE,n_cells
0,iron_willow,"nhits_BF (baseline features, spotlight loss)",0.0357,3.3507e-03,0.0018,1692
1,crimson_fox,"nhits_BFS (BF + spatial lags, spotlight loss)",0.0306,2.6057e-03,0.0014,1692
2,frozen_crown,"nhits_BFC (BF + conflict, spotlight loss)",0.2924,1.2907e-01,0.0711,1692
3,shadow_harbor,"nhits_BFSC (BFS + conflict, spotlight loss)",0.0346,2.9495e-03,0.0016,1692
4,hidden_peak,"tide_BF (baseline features, spotlight loss)",0.0905,1.0103e-02,0.0065,1692
5,golden_rune,"tide_BFS (BF + spatial lags, spotlight loss)",0.0849,9.1690e-03,0.0060,1692
6,silver_drift,"tide_BFC (BF + conflict, spotlight loss)",0.0967,1.1296e-02,0.0070,1692
7,wild_compass,"tide_BFSC (BFS + conflict, spotlight loss)",1.2366,1.5626e+00,0.4244,1692


,model,description,CRPS,MSE,MSLE,n_cells
0,swift_pixel,"nhits_BF (baseline features, MSE loss)",0.0356,3.2568e-03,0.0018,1692
1,broken_spell,"nhits_BFS (BF + spatial lags, MSE loss)",0.0311,2.8355e-03,0.0015,1692
2,rapid_cipher,"nhits_BFC (BF + conflict, MSE loss)",0.0337,3.3032e-03,0.0018,1692
3,quiet_circuit,"nhits_BFSC (BFS + conflict, MSE loss)",0.0327,3.0869e-03,0.0016,1692
4,fuzzy_logic,"tide_BF (baseline features, MSE loss)",0.0510,4.7208e-03,0.0025,1692
5,lucky_packet,"tide_BFS (BF + spatial lags, MSE loss)",0.0511,4.7846e-03,0.0025,1692
6,neon_stack,"tide_BFC (BF + conflict, MSE loss)",0.0328,2.7160e-03,0.0014,1692
7,silent_orbit,"tide_BFSC (BFS + conflict, MSE loss)",0.0371,3.1466e-03,0.0016,1692


### Reading the tables

Start with GDP per capita. Inside a family, compare feature sets at fixed loss: does adding conflict features (BFC, BFSC) help or hurt? Then compare the spotlight table to the MSE table: does the loss choice flip the feature-set ranking? Watch `n_cells`: a row with fewer scored cells is not directly comparable, since it dropped more country-years. Repeat per target; the ranking is not required to agree across outcomes, and where it disagrees that is a finding, not noise.

## Per-origin breakdown

Same metrics, split by rolling origin. The three origins start January 2017, 2018 and 2019, so a target's difficulty changes with the evaluation window; an origin whose years contain heavy conflict (2019 onward covers the war years) scores worse for every model. Use this table to check a pooled ranking is not driven by one easy window.

In [7]:
rows = []
for model in MODEL_ROSTER.model:
    info = MODEL_ROSTER[MODEL_ROSTER.model == model].iloc[0]
    for target in TARGETS:
        for origin, m in results[model][target]["per_origin"].items():
            rows.append({"model": model, "family": info.family,
                         "feature_set": info.feature_set, "loss": info.loss,
                         "target": TARGET_LABELS[target], "origin": origin, **m})
breakdown = pd.DataFrame(rows)
for target in TARGETS:
    display(HTML(f"<h4>Per rolling origin &mdash; {TARGET_LABELS[target]}</h4>"))
    display(breakdown[breakdown.target == TARGET_LABELS[target]]
            .style.format({"CRPS": "{:,.4f}", "MSE": "{:.4e}", "MSLE": "{:.4f}"}))

,model,family,feature_set,loss,target,origin,CRPS,MSE,MSLE,n_cells
0,crimson_fox,nhits,BFS,spotlight,GDP per capita,445,609.7423,1.1430e+06,0.5016,564
1,crimson_fox,nhits,BFS,spotlight,GDP per capita,457,632.4828,1.3874e+06,0.4431,564
2,crimson_fox,nhits,BFS,spotlight,GDP per capita,469,652.6029,1.7166e+06,0.2407,564
24,iron_willow,nhits,BF,spotlight,GDP per capita,445,598.5045,1.8922e+06,0.5181,564
25,iron_willow,nhits,BF,spotlight,GDP per capita,457,757.5105,2.7878e+06,0.4720,564
26,iron_willow,nhits,BF,spotlight,GDP per capita,469,796.6082,2.5517e+06,0.1371,564
48,shadow_harbor,nhits,BFSC,spotlight,GDP per capita,445,518.6337,9.2182e+05,0.4938,564
49,shadow_harbor,nhits,BFSC,spotlight,GDP per capita,457,615.4465,1.5367e+06,0.1835,564
50,shadow_harbor,nhits,BFSC,spotlight,GDP per capita,469,686.0476,1.9198e+06,0.0941,564
72,frozen_crown,nhits,BFC,spotlight,GDP per capita,445,783.1091,2.7778e+06,0.5095,564


,model,family,feature_set,loss,target,origin,CRPS,MSE,MSLE,n_cells
3,crimson_fox,nhits,BFS,spotlight,Tetanus protection,445,1.8702,1.1631e+01,0.0023,564
4,crimson_fox,nhits,BFS,spotlight,Tetanus protection,457,1.8062,1.3569e+01,0.0024,564
5,crimson_fox,nhits,BFS,spotlight,Tetanus protection,469,1.7425,1.0032e+01,0.0021,564
27,iron_willow,nhits,BF,spotlight,Tetanus protection,445,1.8851,1.2619e+01,0.0021,564
28,iron_willow,nhits,BF,spotlight,Tetanus protection,457,1.4975,9.0376e+00,0.0017,564
29,iron_willow,nhits,BF,spotlight,Tetanus protection,469,1.3717,6.3315e+00,0.0014,564
51,shadow_harbor,nhits,BFSC,spotlight,Tetanus protection,445,2.6332,2.2850e+01,0.0038,564
52,shadow_harbor,nhits,BFSC,spotlight,Tetanus protection,457,2.0983,1.4187e+01,0.0027,564
53,shadow_harbor,nhits,BFSC,spotlight,Tetanus protection,469,2.3108,1.5467e+01,0.0028,564
75,frozen_crown,nhits,BFC,spotlight,Tetanus protection,445,2.4008,1.8424e+01,0.0037,564


,model,family,feature_set,loss,target,origin,CRPS,MSE,MSLE,n_cells
6,crimson_fox,nhits,BFS,spotlight,Infant mortality,445,0.8432,2.6280e+01,0.0035,564
7,crimson_fox,nhits,BFS,spotlight,Infant mortality,457,1.0452,5.0738e+01,0.0050,564
8,crimson_fox,nhits,BFS,spotlight,Infant mortality,469,0.7545,1.3120e+01,0.0028,564
30,iron_willow,nhits,BF,spotlight,Infant mortality,445,1.7245,2.7575e+01,0.0087,564
31,iron_willow,nhits,BF,spotlight,Infant mortality,457,1.9410,4.9233e+01,0.0111,564
32,iron_willow,nhits,BF,spotlight,Infant mortality,469,1.9069,3.4154e+01,0.0108,564
54,shadow_harbor,nhits,BFSC,spotlight,Infant mortality,445,1.0646,2.4966e+01,0.0041,564
55,shadow_harbor,nhits,BFSC,spotlight,Infant mortality,457,1.2230,4.3948e+01,0.0053,564
56,shadow_harbor,nhits,BFSC,spotlight,Infant mortality,469,0.9522,1.4185e+01,0.0034,564
78,frozen_crown,nhits,BFC,spotlight,Infant mortality,445,1.7297,2.1545e+01,0.0092,564


,model,family,feature_set,loss,target,origin,CRPS,MSE,MSLE,n_cells
9,crimson_fox,nhits,BFS,spotlight,Life expectancy,445,0.9400,3.2796e+00,0.0010,564
10,crimson_fox,nhits,BFS,spotlight,Life expectancy,457,1.2100,4.1636e+00,0.0013,564
11,crimson_fox,nhits,BFS,spotlight,Life expectancy,469,1.5866,5.9453e+00,0.0015,564
33,iron_willow,nhits,BF,spotlight,Life expectancy,445,1.2688,4.0447e+00,0.0012,564
34,iron_willow,nhits,BF,spotlight,Life expectancy,457,1.5971,7.8793e+00,0.0023,564
35,iron_willow,nhits,BF,spotlight,Life expectancy,469,1.6575,6.3213e+00,0.0019,564
57,shadow_harbor,nhits,BFSC,spotlight,Life expectancy,445,1.0871,2.7324e+00,0.0009,564
58,shadow_harbor,nhits,BFSC,spotlight,Life expectancy,457,1.2893,4.5254e+00,0.0015,564
59,shadow_harbor,nhits,BFSC,spotlight,Life expectancy,469,1.6931,6.2860e+00,0.0017,564
81,frozen_crown,nhits,BFC,spotlight,Life expectancy,445,2.4448,1.2557e+01,0.0030,564


,model,family,feature_set,loss,target,origin,CRPS,MSE,MSLE,n_cells
12,crimson_fox,nhits,BFS,spotlight,Health expenditure p.c.,445,39.4722,5.8770e+03,0.1299,564
13,crimson_fox,nhits,BFS,spotlight,Health expenditure p.c.,457,48.1858,9.2817e+03,0.0288,564
14,crimson_fox,nhits,BFS,spotlight,Health expenditure p.c.,469,66.1277,1.7965e+04,0.0252,564
36,iron_willow,nhits,BF,spotlight,Health expenditure p.c.,445,42.5715,5.5062e+03,0.1643,564
37,iron_willow,nhits,BF,spotlight,Health expenditure p.c.,457,52.8387,1.0376e+04,0.0375,564
38,iron_willow,nhits,BF,spotlight,Health expenditure p.c.,469,70.1551,1.9955e+04,0.0291,564
60,shadow_harbor,nhits,BFSC,spotlight,Health expenditure p.c.,445,42.1413,6.4699e+03,0.1397,564
61,shadow_harbor,nhits,BFSC,spotlight,Health expenditure p.c.,457,53.8907,1.5198e+04,0.0263,564
62,shadow_harbor,nhits,BFSC,spotlight,Health expenditure p.c.,469,69.7286,2.3502e+04,0.0241,564
84,frozen_crown,nhits,BFC,spotlight,Health expenditure p.c.,445,52.6427,9.0257e+03,0.1742,564


,model,family,feature_set,loss,target,origin,CRPS,MSE,MSLE,n_cells
15,crimson_fox,nhits,BFS,spotlight,Stunting,445,0.4198,6.2896e-01,0.0013,564
16,crimson_fox,nhits,BFS,spotlight,Stunting,457,0.3752,5.7401e-01,0.0010,564
17,crimson_fox,nhits,BFS,spotlight,Stunting,469,0.2781,2.9946e-01,0.0007,564
39,iron_willow,nhits,BF,spotlight,Stunting,445,0.7698,1.5358e+00,0.0049,564
40,iron_willow,nhits,BF,spotlight,Stunting,457,0.7718,2.0852e+00,0.0052,564
41,iron_willow,nhits,BF,spotlight,Stunting,469,0.6774,1.2930e+00,0.0045,564
63,shadow_harbor,nhits,BFSC,spotlight,Stunting,445,0.3744,4.0574e-01,0.0011,564
64,shadow_harbor,nhits,BFSC,spotlight,Stunting,457,0.4175,5.2990e-01,0.0013,564
65,shadow_harbor,nhits,BFSC,spotlight,Stunting,469,0.4208,5.1756e-01,0.0015,564
87,frozen_crown,nhits,BFC,spotlight,Stunting,445,0.8854,2.0410e+00,0.0063,564


,model,family,feature_set,loss,target,origin,CRPS,MSE,MSLE,n_cells
18,crimson_fox,nhits,BFS,spotlight,Undernourishment,445,0.7191,4.5251e+00,0.0451,564
19,crimson_fox,nhits,BFS,spotlight,Undernourishment,457,0.7213,2.7531e+00,0.0140,564
20,crimson_fox,nhits,BFS,spotlight,Undernourishment,469,0.8418,3.9525e+00,0.0234,564
42,iron_willow,nhits,BF,spotlight,Undernourishment,445,0.9582,6.1158e+00,0.0644,564
43,iron_willow,nhits,BF,spotlight,Undernourishment,457,0.9752,5.3260e+00,0.0305,564
44,iron_willow,nhits,BF,spotlight,Undernourishment,469,1.0715,4.7950e+00,0.0211,564
66,shadow_harbor,nhits,BFSC,spotlight,Undernourishment,445,0.8206,5.0939e+00,0.0555,564
67,shadow_harbor,nhits,BFSC,spotlight,Undernourishment,457,0.8596,4.2554e+00,0.0212,564
68,shadow_harbor,nhits,BFSC,spotlight,Undernourishment,469,0.9668,5.0451e+00,0.0300,564
90,frozen_crown,nhits,BFC,spotlight,Undernourishment,445,1.0420,6.6887e+00,0.0700,564


,model,family,feature_set,loss,target,origin,CRPS,MSE,MSLE,n_cells
21,crimson_fox,nhits,BFS,spotlight,Liberal democracy,445,0.0303,2.5720e-03,0.0014,564
22,crimson_fox,nhits,BFS,spotlight,Liberal democracy,457,0.0302,2.5763e-03,0.0014,564
23,crimson_fox,nhits,BFS,spotlight,Liberal democracy,469,0.0314,2.6687e-03,0.0014,564
45,iron_willow,nhits,BF,spotlight,Liberal democracy,445,0.0349,3.2904e-03,0.0018,564
46,iron_willow,nhits,BF,spotlight,Liberal democracy,457,0.0357,3.4003e-03,0.0019,564
47,iron_willow,nhits,BF,spotlight,Liberal democracy,469,0.0363,3.3614e-03,0.0018,564
69,shadow_harbor,nhits,BFSC,spotlight,Liberal democracy,445,0.0334,2.8483e-03,0.0015,564
70,shadow_harbor,nhits,BFSC,spotlight,Liberal democracy,457,0.0346,2.8691e-03,0.0015,564
71,shadow_harbor,nhits,BFSC,spotlight,Liberal democracy,469,0.0360,3.1312e-03,0.0017,564
93,frozen_crown,nhits,BFC,spotlight,Liberal democracy,445,0.2988,1.3451e-01,0.0743,564


## Aggregate trajectories

Tables rank; trajectories show shape. One figure per target. Black line: actual annual mean over all countries. Colored lines: one per feature set, averaged over its two loss variants, with the band spanning the two. A model that ranks well on MSE but sits above the black line in the plot is biased high; the plot exposes what a pooled metric hides. Hover gives year and value; drag to zoom a span, double-click to reset.

In [8]:
class TrajectoryPlotter:
    """One aggregate trajectory figure per target: actual annual mean plus one
    line per feature set (mean over its loss variants, band = min/max)."""

    FEATURE_ORDER = ["BF", "BFS", "BFC", "BFSC"]
    FEATURE_COLORS = {"BF": "#1F77B4", "BFS": "#D62728", "BFC": "#2CA02C", "BFSC": "#9467BD"}

    @staticmethod
    def _rgba(hex_color: str, alpha: float) -> str:
        h = hex_color.lstrip("#")
        r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
        return f"rgba({r},{g},{b},{alpha})"

    def figure(self, target: str) -> go.Figure:
        fig = make_subplots(rows=1, cols=2, subplot_titles=("nhits", "tide"))
        plotted = []
        for col, family in ((1, "nhits"), (2, "tide")):
            roster = MODEL_ROSTER[MODEL_ROSTER.family == family]
            annual = {m: results[m][target]["annual"] for m in roster.model}
            any_model = next(iter(annual))
            actual = (annual[any_model].groupby("year", as_index=False)["actual"].mean())
            plotted.append(actual["actual"])
            fig.add_trace(go.Scatter(
                x=actual["year"], y=actual["actual"], name="actual",
                legendgroup="actual", showlegend=(col == 1),
                line=dict(color="black", width=2.5),
                hovertemplate="actual<br>%{x}: %{y:,.2f}<extra></extra>"), row=1, col=col)
            for fs in self.FEATURE_ORDER:
                models = [m for m in roster[roster.feature_set == fs].model if m in annual]
                per_model = pd.concat(
                    [annual[m].groupby("year", as_index=False)["pred"].mean() for m in models])
                mean = per_model.groupby("year", as_index=False)["pred"].mean()
                lo = per_model.groupby("year")["pred"].min()
                hi = per_model.groupby("year")["pred"].max()
                plotted += [mean["pred"], lo, hi]
                color = self.FEATURE_COLORS[fs]
                fig.add_trace(go.Scatter(
                    x=list(lo.index) + list(lo.index)[::-1],
                    y=list(hi) + list(lo)[::-1],
                    fill="toself", fillcolor=self._rgba(color, 0.14),
                    line=dict(width=0), hoverinfo="skip", showlegend=False), row=1, col=col)
                fig.add_trace(go.Scatter(
                    x=mean["year"], y=mean["pred"], name=fs, legendgroup=fs,
                    showlegend=(col == 1),
                    line=dict(color=color, width=2),
                    hovertemplate=f"{fs} ({family}), mean of {len(models)} loss variant(s)"
                                  f"<br>%{{x}}: %{{y:,.2f}}<extra></extra>"), row=1, col=col)
        stack = pd.concat(plotted, ignore_index=True)
        lo, hi = float(stack.min()), float(stack.max())
        pad = 0.06 * (hi - lo)
        fig.update_layout(
            title=dict(text=f"Aggregate {TARGET_LABELS[target]} trajectory", font=dict(size=15)),
            template="plotly_white", width=980, height=430,
            legend=dict(orientation="h", x=0, y=1.06, xanchor="left"),
            margin=dict(t=95, b=55, l=70, r=25), hovermode="x",
        )
        fig.update_xaxes(title_text="year", dtick=1, gridcolor="#E8E8E8")
        fig.update_yaxes(tickformat=",", gridcolor="#E8E8E8", zeroline=False,
                         range=[lo - pad, hi + pad])
        return fig


TRAJ_DIR = RESULTS_DIR / "trajectories"
TRAJ_DIR.mkdir(parents=True, exist_ok=True)
for target in TARGETS:
    fig = TrajectoryPlotter().figure(target)
    fig.write_html(TRAJ_DIR / f"trajectory_{target}.html", include_plotlyjs="cdn")
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
print(f"wrote {len(TARGETS)} trajectory figures to {TRAJ_DIR}")

wrote 8 trajectory figures to /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/trajectories


## Per-model, per-country overlays

Aggregate means can hide a bad country. One figure per model and report country: eight target subplots, each with four lines: that country's monthly y_true (solid black) and the three rolling origins' predictions (blue origin 445, red 457, green 469). No aggregation anywhere.

How to read. Origin lines hugging the black line: model tracks the country. Origin lines that separate: the window matters, usually because conflict hit inside one window. The gap between an origin line and y_true around the origin's start is the nowcast error; the gap at the far end is the long-horizon error. Sixteen models x ten countries, so use the model name in the title to navigate; figures also live as HTML under evaluation_results/calibration_by_model/<model>/.

In [9]:
CALIB_FIG_DIR = RESULTS_DIR / "calibration_by_model"
ORIGIN_COLORS = {"445": "#1F77B4", "457": "#D62728", "469": "#2CA02C"}
TARGET_TICKFORMATS = {
    "lr_gdp_pcap": ",",
    "lr_ttns_zs": ",",
    "lr_imrt_in": ",.1f",
    "lr_le00_in": ",.1f",
    "lr_chex_gd_pcap": ",.0f",
    "lr_stnt_me_zs": ",.1f",
    "lr_defc_zs": ",.1f",
    "lr_v2x_libdem": ".2f",
}


def country_slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


class CountryCalibrationPlotter:
    """One figure per model and report country: eight target subplots, monthly
    y_true (solid black) vs the three rolling origins' predictions."""

    ROWS, COLS = 4, 2

    def __init__(self, model: str, dirs: ModelDirs):
        self.model = model
        self.dirs = dirs
        self.gt = ProcessedGroundTruth(dirs.processed_path())

    def figures(self) -> dict:
        """{country name: (figure, path)} for the ten report countries."""
        origins = self.dirs.calibration_origins()
        first_month = min(int(o.name) for o in origins)
        last_month = max(int(np.load(next(iter(sorted(o.glob("*/identifiers.npz")))))["time"].max())
                         for o in origins)
        pred_cache = {}
        for origin in origins:
            for target in TARGETS:
                if not (origin / target / "identifiers.npz").exists():
                    continue
                time, unit, values = load_pred(origin, target)
                pred_cache[(origin.name, target)] = pd.DataFrame({
                    "month_id": time, "country_id": unit, "pred": values.ravel()})

        figs = {}
        for name, cid in REPORT_COUNTRIES.items():
            fig = make_subplots(rows=self.ROWS, cols=self.COLS,
                                subplot_titles=[TARGET_LABELS[t] for t in TARGETS],
                                horizontal_spacing=0.07, vertical_spacing=0.05)
            for i, target in enumerate(TARGETS):
                row, col = i // self.COLS + 1, i % self.COLS + 1
                fmt = TARGET_TICKFORMATS[target]
                actual = self.gt.actual(target, cid).loc[first_month:last_month]
                fig.add_trace(go.Scatter(
                    x=month_to_ts(actual.index), y=actual.values, name="y_true",
                    legendgroup="y_true", showlegend=(i == 0),
                    mode="lines", line=dict(color="black", width=1.6),
                    hovertemplate=f"{name} y_true<br>%{{x|%b %Y}}: %{{y:{fmt}}}<extra></extra>"),
                    row=row, col=col)
                for origin_name in pred_cache:
                    if (origin_name, target) not in pred_cache:
                        continue
                    s = (pred_cache[(origin_name, target)]
                         .loc[lambda d: d.country_id == cid]
                         .set_index("month_id")["pred"].sort_index())
                    if s.empty:
                        continue
                    fig.add_trace(go.Scatter(
                        x=month_to_ts(s.index), y=s.values, name=origin_name,
                        legendgroup=origin_name, showlegend=(i == 0),
                        mode="lines",
                        line=dict(color=ORIGIN_COLORS.get(origin_name, "gray"), width=1.2),
                        hovertemplate=f"{name} origin {origin_name}"
                                      f"<br>%{{x|%b %Y}}: %{{y:{fmt}}}<extra></extra>"),
                        row=row, col=col)
                fig.update_yaxes(tickformat=fmt, gridcolor="#ECECEC", row=row, col=col)
            fig.update_layout(
                title=dict(text=f"{self.model} &mdash; {name}, monthly y_true vs origins",
                           font=dict(size=15)),
                template="plotly_white", width=1050, height=1250,
                legend=dict(orientation="h", x=0, y=1.005, xanchor="left"),
                margin=dict(t=85, b=45, l=60, r=20), hovermode="closest",
            )
            fig.update_xaxes(ticks="outside", tickformat="%Y", dtick="M24", gridcolor="#ECECEC")
            model_dir = CALIB_FIG_DIR / self.model
            model_dir.mkdir(parents=True, exist_ok=True)
            path = model_dir / f"{country_slug(name)}.html"
            fig.write_html(path, include_plotlyjs="cdn")
            figs[name] = (fig, path)
        return figs


for model in MODEL_ROSTER.model:
    plotter = CountryCalibrationPlotter(model, MODEL_DIRS[model])
    figs = plotter.figures()
    print(f"{model}: wrote {len(figs)} country figures")

crimson_fox: wrote 10 country figures
iron_willow: wrote 10 country figures
shadow_harbor: wrote 10 country figures
frozen_crown: wrote 10 country figures
broken_spell: wrote 10 country figures
swift_pixel: wrote 10 country figures
quiet_circuit: wrote 10 country figures
rapid_cipher: wrote 10 country figures
golden_rune: wrote 10 country figures
hidden_peak: wrote 10 country figures
wild_compass: wrote 10 country figures
silver_drift: wrote 10 country figures
lucky_packet: wrote 10 country figures
fuzzy_logic: wrote 10 country figures
silent_orbit: wrote 10 country figures
neon_stack: wrote 10 country figures


## Report figures 3 to 5, paper version

Layout of the paper's figures 3 to 5: one panel per report country, two per row, 1990 to now. Black diamonds: observed outcome from the viewser fetch (runs to April 2026). Green line, right axis: observed conflict fatalities, state-based plus one-sided plus non-state summed. Thin red line: forecast.

One model per figure set, not an ensemble average. This set uses frozen_crown (nhits, BFC features, spotlight loss), the paper's preferred model, forecasting from January 2026 through October 2029 out of forecasting/553. Nothing in a panel is averaged over models or countries; the next section repeats the same three figures for every one of the sixteen models.

The paper's yellow counterfactual line is the same model rerun with conflict history zeroed (report section 4.5). No stored run does that yet, so the yellow line and impact figures 6 and 7 wait until the forecasting stage runs with the pipeline's counterfactual flag. Override runs, when present, show up in the audit above but not in these panels.

In [10]:
REPORT_FIG_DIR = RESULTS_DIR / "figures"
REPORT_FIG_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_COLOR, CONFLICT_COLOR = "red", "green"
OUTCOME_AXES = {
    TARGET: dict(label="GDP per capita (USD)", tickformat=",", hoverfmt=",.0f"),
    "lr_imrt_in": dict(label="Infant mortality rate", tickformat=",", hoverfmt=",.1f"),
    "lr_v2x_libdem": dict(label="Liberal democracy index", tickformat=".2f", hoverfmt=",.3f"),
}


class ObservedOutcomes:
    """Observed monthly outcome, conflict and population series per country,
    read from the informed model's processed parquet under models2, the same
    file the evaluator scores against."""

    def __init__(self, parquet: Path):
        df = pd.read_parquet(parquet).reset_index()
        cols = [c for c in (TARGET, "lr_imrt_in", "lr_v2x_libdem",
                            *FATALITY_COLUMNS, "lr_pop_totl") if c in df.columns]
        self.wide = {c: df.pivot(index="month_id", columns="country_id", values=c) for c in cols}

    def series(self, column: str, country_id: int) -> pd.Series:
        return self.wide[column][country_id].dropna()

    def fatalities(self, country_id: int) -> pd.Series:
        return sum(self.wide[c][country_id] for c in FATALITY_COLUMNS).dropna()


class ReportFigureFactory:
    """Report figures 3 to 5: observed diamonds, conflict line on a green right
    axis, thin red forecast line from the plain forecasting run."""

    ROWS, COLS = 5, 2

    def __init__(self, observed: ObservedOutcomes, forecast_run: Path):
        self.observed = observed
        self.forecast_run = forecast_run

    def _panel(self, fig, row, col, name, country_id, outcome, show_forecast, letter,
               counterfactual_run=None):
        first = letter == "a"
        fmt = OUTCOME_AXES[outcome]["hoverfmt"]
        obs = self.observed.series(outcome, country_id)
        fig.add_trace(go.Scatter(
            x=month_to_ts(obs.index), y=obs.values, name="observed",
            legendgroup="observed", showlegend=first,
            mode="lines+markers", marker=dict(symbol="diamond", size=4),
            line=dict(color="black", width=1.1),
            hovertemplate=f"%{{x|%b %Y}}: %{{y:{fmt}}}<extra>observed</extra>"),
            row=row, col=col)
        if show_forecast:
            time, unit, values = load_pred(self.forecast_run, outcome)
            df = pd.DataFrame({"month_id": time, "country_id": unit, "pred": values.ravel()})
            s = (df.loc[df.country_id == country_id]
                 .set_index("month_id")["pred"].sort_index())
            fig.add_trace(go.Scatter(
                x=month_to_ts(s.index), y=s.values, name="forecast (BFC)",
                legendgroup="forecast", showlegend=first,
                mode="lines", line=dict(color=FORECAST_COLOR, width=1.0),
                hovertemplate=f"%{{x|%b %Y}}: %{{y:{fmt}}}<extra>forecast (BFC)</extra>"),
                row=row, col=col)
        if counterfactual_run is not None and (counterfactual_run / outcome / "identifiers.npz").exists():
            time, unit, values = load_pred(counterfactual_run, outcome)
            df = pd.DataFrame({"month_id": time, "country_id": unit, "pred": values.ravel()})
            s = (df.loc[df.country_id == country_id]
                 .set_index("month_id")["pred"].sort_index())
            fig.add_trace(go.Scatter(
                x=month_to_ts(s.index), y=s.values, name="counterfactual (no conflict)",
                legendgroup="counterfactual", showlegend=first,
                mode="lines", line=dict(color="gold", width=1.2),
                hovertemplate=f"%{{x|%b %Y}}: %{{y:{fmt}}}<extra>counterfactual</extra>"),
                row=row, col=col)
        fat = self.observed.fatalities(country_id)
        fig.add_trace(go.Scatter(
            x=month_to_ts(fat.index), y=fat.values, name="conflict fatalities",
            legendgroup="conflict", showlegend=first,
            mode="lines", line=dict(color=CONFLICT_COLOR, width=1.0),
            hovertemplate="%{x|%b %Y}: %{y:,.0f}<extra>fatalities</extra>"),
            row=row, col=col, secondary_y=True)

    def outcome_panels(self, outcome, title, filename, show_forecast, out_dir=None,
                       counterfactual_run=None):
        out_dir = out_dir or REPORT_FIG_DIR
        specs = [[{"secondary_y": True}] * self.COLS for _ in range(self.ROWS)]
        subplot_titles = [f"({chr(97 + i)}) {name}" for i, name in enumerate(REPORT_COUNTRIES)]
        fig = make_subplots(rows=self.ROWS, cols=self.COLS, specs=specs,
                            subplot_titles=subplot_titles,
                            horizontal_spacing=0.09, vertical_spacing=0.045)
        for i, (name, cid) in enumerate(REPORT_COUNTRIES.items()):
            self._panel(fig, i // self.COLS + 1, i % self.COLS + 1,
                        name, cid, outcome, show_forecast, chr(97 + i),
                        counterfactual_run=counterfactual_run)
        fig.update_layout(
            title=dict(text=title, font=dict(size=16)),
            template="plotly_white", width=1000, height=1500,
            legend=dict(orientation="h", x=0, y=1.01, xanchor="left"),
            margin=dict(t=90, b=45, l=60, r=20), hovermode="closest",
        )
        fig.update_xaxes(ticks="outside", tickformat="%Y", dtick="M120", gridcolor="#ECECEC")
        fig.update_yaxes(tickformat=OUTCOME_AXES[outcome]["tickformat"],
                         gridcolor="#ECECEC", secondary_y=False)
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, secondary_y=True)
        out_dir.mkdir(parents=True, exist_ok=True)
        out = out_dir / filename
        fig.write_html(out, include_plotlyjs="cdn")
        return fig, out


# forecasting-window processed table carries monthly actuals/nowcasts through
# the forecasting horizon, so the observed line meets the forecast with no gap
observed = ObservedOutcomes(MODEL_DIRS[INFORMED_MODEL].processed_path("forecasting"))
counterfactual_run = MODEL_DIRS[INFORMED_MODEL].counterfactual_forecasting()
print("counterfactual run:", counterfactual_run.name if counterfactual_run else "none yet")
plain_run = MODEL_DIRS[INFORMED_MODEL].plain_forecasting()
print("plain forecasting run:", plain_run.name if plain_run else "MISSING")
overrides = MODEL_DIRS[INFORMED_MODEL].override_forecasting()
print("override runs present:", {o: p.name for o, p in overrides.items()} or "none yet")

info = MODEL_ROSTER[MODEL_ROSTER.model == INFORMED_MODEL].iloc[0]
paper_tag = f"{INFORMED_MODEL} ({info.family} {info.feature_set}, {info.loss} loss)"
factory = ReportFigureFactory(observed, plain_run)
figures = {
    "fig3": factory.outcome_panels(
        TARGET, f"Observed and predicted GDP per capita (USD), selected countries &mdash; forecast: {paper_tag}, yellow: counterfactual",
        "fig3_gdp_per_capita.html", show_forecast=True, counterfactual_run=counterfactual_run),
    "fig4": factory.outcome_panels(
        "lr_imrt_in", f"Observed and predicted infant mortality rate, selected countries &mdash; forecast: {paper_tag}, yellow: counterfactual",
        "fig4_infant_mortality.html", show_forecast=True, counterfactual_run=counterfactual_run),
    "fig5": factory.outcome_panels(
        "lr_v2x_libdem", f"Observed and predicted liberal democracy index, selected countries &mdash; forecast: {paper_tag}, yellow: counterfactual",
        "fig5_liberal_democracy.html", show_forecast=True, counterfactual_run=counterfactual_run),
}
for key, (fig, path) in figures.items():
    display(HTML(f"<h4>{key}</h4>"))
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    print(f"saved {path}  ({path.stat().st_size/1024:.0f} KB)")

counterfactual run: 553_counterfactual
plain forecasting run: 553
override runs present: none yet


saved /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/fig3_gdp_per_capita.html  (403 KB)


saved /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/fig4_infant_mortality.html  (403 KB)


saved /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/fig5_liberal_democracy.html  (411 KB)


## Report figures 3 to 5, every model

Same three figures, forecast line swapped per model: sixteen rows of three, all sixteen models, so a country panel can be compared across models directly. The layout, colors and axes match the paper version; only the thin red line changes. Saved under evaluation_results/figures/by_model/<model>/.

How to use this section. Pick a country you know, pick a target, and walk the sixteen red lines: models whose forecast bends with the observed history track that country; models that flatten or drift are imposing their average-country shape. Cross-check against the tables above: a model can score well pooled and still misfit the countries you care about, because pooled metrics are dominated by the many calm countries.

In [11]:
PER_MODEL_FIG_DIR = REPORT_FIG_DIR / "by_model"

for model in MODEL_ROSTER.model:
    run = MODEL_DIRS[model].plain_forecasting()
    if run is None:
        print(f"{model}: no plain forecasting run, skipping")
        continue
    info = MODEL_ROSTER[MODEL_ROSTER.model == model].iloc[0]
    tag = f"{model} ({info.family} {info.feature_set}, {info.loss} loss)"
    factory = ReportFigureFactory(observed, run)
    cf_run = MODEL_DIRS[model].counterfactual_forecasting()
    cf_note = ", yellow: counterfactual" if cf_run else ""
    model_dir = PER_MODEL_FIG_DIR / model
    model_figs = {
        "fig3": factory.outcome_panels(
            TARGET, f"Observed and predicted GDP per capita (USD) &mdash; forecast: {tag}{cf_note}",
            "fig3_gdp_per_capita.html", show_forecast=True, out_dir=model_dir,
            counterfactual_run=cf_run),
        "fig4": factory.outcome_panels(
            "lr_imrt_in", f"Observed and predicted infant mortality rate &mdash; forecast: {tag}{cf_note}",
            "fig4_infant_mortality.html", show_forecast=True, out_dir=model_dir,
            counterfactual_run=cf_run),
        "fig5": factory.outcome_panels(
            "lr_v2x_libdem", f"Observed and predicted liberal democracy index &mdash; forecast: {tag}{cf_note}",
            "fig5_liberal_democracy.html", show_forecast=True, out_dir=model_dir,
            counterfactual_run=cf_run),
    }
    for key, (fig, path) in model_figs.items():
        display(HTML(f"<h5>{model} &mdash; {key}</h5>"))
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    print(f"{model}: wrote 3 figures under {PER_MODEL_FIG_DIR / model}")

crimson_fox: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/crimson_fox


iron_willow: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/iron_willow


shadow_harbor: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/shadow_harbor


frozen_crown: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/frozen_crown


broken_spell: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/broken_spell


swift_pixel: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/swift_pixel


... (6 outputs left out)

lucky_packet: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/lucky_packet


fuzzy_logic: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/fuzzy_logic


silent_orbit: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/silent_orbit


neon_stack: wrote 3 figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/by_model/neon_stack


## Report figures, calibration version

Same layout as the report figures, but the predictions come from the calibration runs, so the forecast lines sit on top of history the notebook can score: the three thin lines are the rolling origins starting January 2017, 2018 and 2019 (red 445, orange 457, dark red 469), each 36 months long and overlapping in the middle. Black diamonds stay the observed outcome, green stays conflict fatalities on the right axis.

This is the fit view the forecasting figures cannot give: here every red pixel can be compared against a diamond. A model whose origin lines hug the diamonds is tracking history; a model that only looks right from 2026 onward is untested. One figure set per model, three outcomes, saved under evaluation_results/figures/calibration/<model>/.

In [12]:
CALIB_REPORT_DIR = REPORT_FIG_DIR / "calibration"

CALIB_ORIGIN_COLORS = {"445": "#D62728", "457": "#FF7F0E", "469": "#B22222"}


class CalibrationReportFigures:
    """Report-figure panels with the calibration rolling origins drawn as thin
    lines over the observed history, per model, per outcome."""

    ROWS, COLS = 5, 2

    def __init__(self, model: str, observed: ObservedOutcomes):
        self.model = model
        self.observed = observed
        self.origins = MODEL_DIRS[model].calibration_origins()

    def _panel(self, fig, row, col, name, country_id, outcome, letter):
        first = letter == "a"
        fmt = OUTCOME_AXES[outcome]["hoverfmt"]
        obs = self.observed.series(outcome, country_id)
        fig.add_trace(go.Scatter(
            x=month_to_ts(obs.index), y=obs.values, name="observed",
            legendgroup="observed", showlegend=first,
            mode="lines+markers", marker=dict(symbol="diamond", size=4),
            line=dict(color="black", width=1.1),
            hovertemplate=f"%{{x|%b %Y}}: %{{y:{fmt}}}<extra>observed</extra>"),
            row=row, col=col)
        for origin in self.origins:
            if not (origin / outcome / "identifiers.npz").exists():
                continue
            time, unit, values = load_pred(origin, outcome)
            df = pd.DataFrame({"month_id": time, "country_id": unit, "pred": values.ravel()})
            s = (df.loc[df.country_id == country_id]
                 .set_index("month_id")["pred"].sort_index())
            color = CALIB_ORIGIN_COLORS.get(origin.name, "gray")
            fig.add_trace(go.Scatter(
                x=month_to_ts(s.index), y=s.values, name=f"origin {origin.name}",
                legendgroup=f"origin{origin.name}", showlegend=first,
                mode="lines", line=dict(color=color, width=1.0),
                hovertemplate=f"{name} origin {origin.name}"
                              f"<br>%{{x|%b %Y}}: %{{y:{fmt}}}<extra></extra>"),
                row=row, col=col)
        fat = self.observed.fatalities(country_id)
        fig.add_trace(go.Scatter(
            x=month_to_ts(fat.index), y=fat.values, name="conflict fatalities",
            legendgroup="conflict", showlegend=first,
            mode="lines", line=dict(color=CONFLICT_COLOR, width=1.0),
            hovertemplate="%{x|%b %Y}: %{y:,.0f}<extra>fatalities</extra>"),
            row=row, col=col, secondary_y=True)

    def outcome_panels(self, outcome, title, filename, out_dir):
        specs = [[{"secondary_y": True}] * self.COLS for _ in range(self.ROWS)]
        subplot_titles = [f"({chr(97 + i)}) {name}" for i, name in enumerate(REPORT_COUNTRIES)]
        fig = make_subplots(rows=self.ROWS, cols=self.COLS, specs=specs,
                            subplot_titles=subplot_titles,
                            horizontal_spacing=0.09, vertical_spacing=0.045)
        for i, (name, cid) in enumerate(REPORT_COUNTRIES.items()):
            self._panel(fig, i // self.COLS + 1, i % self.COLS + 1,
                        name, cid, outcome, chr(97 + i))
        fig.update_layout(
            title=dict(text=title, font=dict(size=15)),
            template="plotly_white", width=1000, height=1500,
            legend=dict(orientation="h", x=0, y=1.01, xanchor="left"),
            margin=dict(t=90, b=45, l=60, r=20), hovermode="closest",
        )
        fig.update_xaxes(ticks="outside", tickformat="%Y", dtick="M120", gridcolor="#ECECEC")
        fig.update_yaxes(tickformat=OUTCOME_AXES[outcome]["tickformat"],
                         gridcolor="#ECECEC", secondary_y=False)
        fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, secondary_y=True)
        out_dir.mkdir(parents=True, exist_ok=True)
        out = out_dir / filename
        fig.write_html(out, include_plotlyjs="cdn")
        return fig, out


for model in MODEL_ROSTER.model:
    info = MODEL_ROSTER[MODEL_ROSTER.model == model].iloc[0]
    tag = f"{model} ({info.family} {info.feature_set}, {info.loss} loss)"
    builder = CalibrationReportFigures(model, observed)
    out_dir = CALIB_REPORT_DIR / model
    model_figs = {
        "fig3": builder.outcome_panels(
            TARGET, f"Observed and calibration-predicted GDP per capita (USD) &mdash; {tag}",
            "fig3_gdp_per_capita.html", out_dir),
        "fig4": builder.outcome_panels(
            "lr_imrt_in", f"Observed and calibration-predicted infant mortality rate &mdash; {tag}",
            "fig4_infant_mortality.html", out_dir),
        "fig5": builder.outcome_panels(
            "lr_v2x_libdem", f"Observed and calibration-predicted liberal democracy index &mdash; {tag}",
            "fig5_liberal_democracy.html", out_dir),
    }
    for key, (fig, path) in model_figs.items():
        display(HTML(f"<h5>{model} &mdash; calibration {key}</h5>"))
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    print(f"{model}: wrote 3 calibration figures under {out_dir}")

crimson_fox: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/crimson_fox


iron_willow: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/iron_willow


shadow_harbor: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/shadow_harbor


frozen_crown: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/frozen_crown


broken_spell: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/broken_spell


swift_pixel: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/swift_pixel


... (6 outputs left out)

lucky_packet: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/lucky_packet


fuzzy_logic: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/fuzzy_logic


silent_orbit: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/silent_orbit


neon_stack: wrote 3 calibration figures under /Users/dylanpinheiro/Library/CloudStorage/Dropbox/ReplicationData/Impact/models2/evaluation_results/figures/calibration/neon_stack


### What the results say

**Metric interpretation, one paragraph each.** CRPS here is the mean absolute error of yearly predictions: the typical miss, in the target's own units. MSE squares errors, so a handful of badly missed countries dominate it; a model can win MSE while losing CRPS if it avoids the worst blowups but misses moderately everywhere. MSLE scores log(1+x), which makes it a relative error: missing by 200 on a 300-dollar GDP matters as much as missing by 20,000 on a 30,000-dollar one, so MSLE favors models that track poor countries. `n_cells` guards the comparison: rows with fewer scored country-years dropped more data and are not directly comparable. Rankings below use CRPS rank per target, averaged over the eight targets.

**Headline.** nhits wins everywhere. The four nhits BFS/BFSC models take the top four mean ranks (crimson_fox 2.25, broken_spell 2.50, shadow_harbor 4.13, quiet_circuit 4.88); every tide model sits at 8 or worse except neon_stack (8.00). Within nhits, spatial lags help and conflict features hurt: BFS leads, BFSC trails it slightly, BF and BFC trail further. Tide inverts: its best models are the conflict-fed ones (neon_stack, tide BFC MSE, rank 5 on GDP and best overall tide), while tide BF/BFS are the four worst models of the entire roster. Training loss barely matters for nhits (mean rank 5.69 MSE vs 5.94 spotlight) and gives spotlight a small edge for tide (10.81 vs 11.56).

**Per model, what it did well and badly.**

- crimson_fox (nhits BFS, spotlight): best overall. First on infant mortality, stunting and liberal democracy, second on life expectancy, health expenditure and undernourishment, third on GDP. Weak spot: tetanus, sixth. Balanced across outcomes, no collapse anywhere.
- broken_spell (nhits BFS, MSE): second overall and the single-target winner: first on GDP, tetanus, health expenditure and undernourishment. Mid-table on life expectancy and stunting. Pairing it with crimson_fox shows the loss choice moves the ranking by one or two places, not by kind.
- shadow_harbor / quiet_circuit (nhits BFSC): solid but never best; adding conflict features and lags to nhits costs a little accuracy versus pure BFS. quiet_circuit posts the best GDP MSLE (0.182), so it tracks poor countries' GDP best in relative terms.
- iron_willow / swift_pixel (nhits BF): mid-table on everything. Population alone is a serviceable baseline but the spatial lag is worth real accuracy.
- rapid_cipher / frozen_crown (nhits BFC): conflict features without lags hurt nhits the most. frozen_crown, the paper's preferred model, ranks 13 of 16 on GDP and 15 on liberal democracy in this calibration window; the paper's choice rested on the earlier single-target runs, and this multi-target evaluation does not support it. Flag for the paper: either re-select the preferred model or justify frozen_crown on forecasting-window grounds the calibration metrics do not cover.
- neon_stack / silver_drift (tide BFC): the respectable tide models. neon_stack is fifth on GDP, third on health expenditure, fourth on democracy; it is the counterexample that tide is not hopeless once it sees conflict. silver_drift similar but weaker.
- wild_compass (tide BFSC): mid on most targets, then blows up on liberal democracy: CRPS 1.24 against a roster median near 0.03, forty times too high, meaning at least one country's democracy forecast diverges badly. Check its fig5 panels before trusting it on institutions.
- silent_orbit (tide BFSC, MSE): consistently below the nhits pack on every target, never catastrophic except tetanus.
- golden_rune / lucky_packet (tide BFS) and fuzzy_logic / hidden_peak (tide BF): the bottom four. hidden_peak's tetanus forecast collapsed outright (CRPS 83 versus a roster median near 3; MSE 12,315) and golden_rune's undernourishment blew up the same way (CRPS 19, MSE 868). These blowups, not steady mediocrity, are what sink the tide baselines; their overlay figures show which countries drove it.

**Two caveats.** These scores come from the 2017-2021 evaluation windows of the rolling origins; the forecasting window (2026-2029) can reorder models, which is exactly what the per-model report figures above let you check. And a model that wins pooled CRPS is not automatically the right model for a specific country: pooled ranks are dominated by the many calm countries, so for conflict countries prefer the per-country overlays and the per-origin breakdown.

### Reading the figures, and what is still open

Figures 3 to 5 follow the report's layout. Where a red forecast line bends away from the observed diamonds, that bend is that model's story for that country; the green conflict line is the candidate explanation. The results section above tells you which model to trust on which target before you read the lines.

Three things stay open, on purpose. Counterfactual runs (conflict history zeroed, report section 4.5) do not exist yet, so the yellow counterfactual lines and impact figures 6 and 7 wait. Override forecasting runs are not in the stores yet; when they land they organize themselves as forecasting/561, 562, 563 and the audit cell shows them. And CRPS here is the mean absolute error, a consequence of the single stored sample per cell, not a bug.